# Blif to Graph

This code is a simple parser for the BLIF format. It reads a BLIF file and creates a graph with the gates and connections between them. This graph is used to generate a cuda kernel that evaluates the circuit.

In [101]:
import os
import networkx as nx
import numpy as np
import math

In [102]:
N    = 4
PATH = f'{os.getcwd()}/results/blif_generator'
# Verify if the directory exists
if not os.path.exists(PATH):
    os.makedirs(PATH)



print(PATH)

/app/results/blif_generator


## BLIF parser

In [103]:

def binary_to_decimal(binary_str):
    return int(binary_str.replace('-', '0'), 2)

def parse_blif(file_path):
    G = nx.DiGraph()
    
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    inputs, outputs = [], []
    parsing_names = False
    current_node = None
    node_data = {}
    input_order = {}
    
    for line in lines:
        tokens = line.strip().split()

        
        if tokens[0] == ".inputs":
            inputs = tokens[1:]
            G.add_nodes_from(inputs, type="input")
        elif tokens[0] == ".outputs":
            outputs = tokens[1:]
            G.add_nodes_from(outputs, type="output")
        elif tokens[0] == ".names":
            parsing_names = True
            current_node = tokens[-1]
            dependencies = tokens[1:-1]
            node_data[current_node] = ""
            input_order[current_node] = dependencies  # Armazena a ordem das entradas

            G.nodes[current_node]['input_order'] = dependencies
            G.nodes[current_node]['truth_value'] = None

            for dep in dependencies:
                if dep not in G.nodes():
                    G.add_node(dep, type="intermediary", input_order=[], truth_value=None)
                G.add_edge(dep, current_node)
        elif parsing_names and current_node is not None and len(tokens) > 1:
            output_value = tokens[-1]
            if output_value.isdigit():  # Verifica se o valor de saída é realmente um número
                node_data[current_node] += output_value  # Concatena os valores de saída
    
    for node, binary_str in node_data.items():
        G.nodes[node]['truth_value'] = int(binary_str, 2) if binary_str else None  # Converte a string binária para decimal
    
    return G

# Exemplo de uso
graph = parse_blif(f"{PATH}/somador.blif")
graph.nodes(data=True)


NodeDataView({'x0': {'type': 'input'}, 'x1': {'type': 'input'}, 'x2': {'type': 'input'}, 'x3': {'type': 'input'}, 'x4': {'type': 'input'}, 'x5': {'type': 'input'}, 'z0': {'type': 'output', 'input_order': ['new_n13', 'x0', 'new_n14', 'new_n15'], 'truth_value': 3}, 'z1': {'type': 'output', 'input_order': ['new_n17', 'x0', 'x3'], 'truth_value': 15}, 'z2': {'type': 'output', 'input_order': ['x2', 'x5', 'x1', 'x4'], 'truth_value': 63}, 'z3': {'type': 'output', 'input_order': ['x2', 'x5'], 'truth_value': 3}, 'new_n13': {'type': 'intermediary', 'input_order': ['x1', 'x2', 'x5', 'x4', 'x3'], 'truth_value': 0}, 'new_n14': {'type': 'intermediary', 'input_order': ['x0', 'x1', 'x4', 'x5', 'x2'], 'truth_value': 15}, 'new_n15': {'type': 'intermediary', 'input_order': ['x4', 'x1', 'x2', 'x3'], 'truth_value': 3}, 'new_n17': {'type': 'intermediary', 'input_order': ['x1', 'x4', 'x2', 'x5'], 'truth_value': 7}})

In [104]:
import networkx as nx
from itertools import product

def binary_to_decimal(binary_str):
    return int(binary_str, 2)

def pla_to_binary_vector(pla_table, num_vars):
    """
    Converte uma tabela PLA para um vetor binário.
    """
    vector_size = 2 ** num_vars
    binary_vector = [0] * vector_size
    
    for entry in pla_table:
        pattern, output = "".join(entry["input"]), entry["output"]
        if output == '1':
            positions = [i for i, char in enumerate(pattern) if char == '-']
            base_pattern = list(pattern)
            
            for replacements in product('01', repeat=len(positions)):
                for pos, char in zip(positions, replacements):
                    base_pattern[pos] = char
                index = int(''.join(base_pattern), 2)
                binary_vector[index] = 1
    
    # Inverte o vetor para que a ordem dos bits seja a mesma da tabela PLA
    binary_vector.reverse()
    return binary_vector

def parse_blif(file_path):
    G = nx.DiGraph()
    
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    inputs, outputs = [], []
    parsing_names = False
    current_node = None
    node_data = {}
    input_order = {}
    
    for line in lines:
        tokens = line.strip().split()

        if tokens[0] == ".inputs":
            inputs = tokens[1:]
            G.add_nodes_from(inputs, type="input")
        elif tokens[0] == ".outputs":
            outputs = tokens[1:]
            G.add_nodes_from(outputs, type="output")
        elif tokens[0] == ".names":
            parsing_names = True
            current_node = tokens[-1]
            dependencies = tokens[1:-1]
     
            node_data[current_node] = []
   
            input_order[current_node] = dependencies

            G.nodes[current_node]['input_order'] = dependencies
            G.nodes[current_node]['truth_value'] = None

            for dep in dependencies:
                if dep not in G.nodes():
                    G.add_node(dep, type="intermediary", input_order=[], truth_value=None)
                G.add_edge(dep, current_node)
        elif parsing_names and current_node is not None and len(tokens) > 1:
            input_values = tokens[:-1]
            output_value = tokens[-1]
            node_data[current_node].append({"input": input_values, "output": output_value})
    
    for node, partial_truth_table in node_data.items():
        num_vars = len(input_order[node])
        full_truth_table = pla_to_binary_vector(partial_truth_table, num_vars)
        G.nodes[node]['truth_value'] = binary_to_decimal("".join(map(str, full_truth_table)))
    
    return G

# Exemplo de uso
graph = parse_blif(f"{PATH}/somador.blif")
graph.nodes(data=True)


NodeDataView({'x0': {'type': 'input'}, 'x1': {'type': 'input'}, 'x2': {'type': 'input'}, 'x3': {'type': 'input'}, 'x4': {'type': 'input'}, 'x5': {'type': 'input'}, 'z0': {'type': 'output', 'input_order': ['new_n13', 'x0', 'new_n14', 'new_n15'], 'truth_value': 8946}, 'z1': {'type': 'output', 'input_order': ['new_n17', 'x0', 'x3'], 'truth_value': 150}, 'z2': {'type': 'output', 'input_order': ['x2', 'x5', 'x1', 'x4'], 'truth_value': 38502}, 'z3': {'type': 'output', 'input_order': ['x2', 'x5'], 'truth_value': 6}, 'new_n13': {'type': 'intermediary', 'input_order': ['x1', 'x2', 'x5', 'x4', 'x3'], 'truth_value': 0}, 'new_n14': {'type': 'intermediary', 'input_order': ['x0', 'x1', 'x4', 'x5', 'x2'], 'truth_value': 2192179754}, 'new_n15': {'type': 'intermediary', 'input_order': ['x4', 'x1', 'x2', 'x3'], 'truth_value': 43144}, 'new_n17': {'type': 'intermediary', 'input_order': ['x1', 'x4', 'x2', 'x5'], 'truth_value': 63616}})

## CUDA kernel generation

In [105]:


def generate_cpp_equation(node, truth_value, input_order):
    """Gera o código C++ para a equação booleana do nó."""
    # Convertendo o valor da truth_value para decimal
    truth_value_decimal = truth_value
    
    # Inicializando a equação
    equation = f"int {node} = {truth_value_decimal} >> ("
    
    # Gerando os deslocamentos para input_order
    terms = []
    len_order = len(input_order)
    for i in range(len_order):
        term = f"({input_order[i]} << {len_order - 1 - i})"
        terms.append(term)
    
    # Adicionando os termos à equação
    equation += " + ".join(terms) + ") & 1;"
    
    return equation

def generate_code_from_graph(G):
    """Gera o código C++ para todos os nós intermediários do grafo."""
    code = []

    # Intera sobre os nós de input para inicializar
    for node, data in G.nodes(data=True):
        if data.get('type') == 'input':
            code.append(f"int {node};")
    
    # Itera sobre todos os nós no grafo
    for node, data in G.nodes(data=True):
        if data.get('type') == 'intermediary':  # Processa apenas nós intermediários
            truth_value = data.get('truth_value', '0')  # Valor de truth_value do nó
            input_order = data.get('input_order', [])
            
            # Gerar o código C++ para o nó
            cpp_equation = generate_cpp_equation(node, truth_value, input_order)
            code.append(cpp_equation)
    
    for node, data in G.nodes(data=True):
        if data.get('type') == 'output':  # Processa apenas nós de saída
            truth_value = data.get('truth_value', '0')  # Valor de truth_value do nó
            input_order = data.get('input_order', [])
            
            # Gerar o código C++ para o nó
            cpp_equation = generate_cpp_equation(node, truth_value, input_order)
            code.append(cpp_equation)
    
    return code



# Gerar o código C++ a partir do grafo
cpp_code = generate_code_from_graph(graph)

# Exibir o código gerado
for line in cpp_code:
    print(line)


int x0;
int x1;
int x2;
int x3;
int x4;
int x5;
int new_n13 = 0 >> ((x1 << 4) + (x2 << 3) + (x5 << 2) + (x4 << 1) + (x3 << 0)) & 1;
int new_n14 = 2192179754 >> ((x0 << 4) + (x1 << 3) + (x4 << 2) + (x5 << 1) + (x2 << 0)) & 1;
int new_n15 = 43144 >> ((x4 << 3) + (x1 << 2) + (x2 << 1) + (x3 << 0)) & 1;
int new_n17 = 63616 >> ((x1 << 3) + (x4 << 2) + (x2 << 1) + (x5 << 0)) & 1;
int z0 = 8946 >> ((new_n13 << 3) + (x0 << 2) + (new_n14 << 1) + (new_n15 << 0)) & 1;
int z1 = 150 >> ((new_n17 << 2) + (x0 << 1) + (x3 << 0)) & 1;
int z2 = 38502 >> ((x2 << 3) + (x5 << 2) + (x1 << 1) + (x4 << 0)) & 1;
int z3 = 6 >> ((x2 << 1) + (x5 << 0)) & 1;


In [106]:
import os
import networkx as nx

def generate_cuda_code(G):
    # Count inputs and outputs
    inputs = [node for node, data in G.nodes(data=True) if data.get('type') == 'input']
    outputs = [node for node, data in G.nodes(data=True) if data.get('type') == 'output']
    num_inputs = len(inputs)
    num_outputs = len(outputs)
    
    # Get nodes in topological order
    ordered_nodes = list(nx.topological_sort(G))
    
    cuda_code = """
#include <stdio.h>
#include <cuda_runtime.h>
#include <stdlib.h>

// Utility function to check CUDA errors
#define CHECK_CUDA_ERROR(call) { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        printf("CUDA error: %s\\n", cudaGetErrorString(err)); \
        exit(1); \
    } \
}

__global__ void evaluate_circuit(int *output) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int num_combinations = 1 << """ + str(num_inputs) + """;
    
    if (idx < num_combinations) {
        int x = idx;
"""
    
    # Initialize all variables to 0
    all_nodes = set(G.nodes())
    cuda_code += "\n        // Initialize all variables\n"
    for node in all_nodes:
        cuda_code += f"        int {node} = 0;\n"
    
    cuda_code += "\n        // Set input values\n"
    # Input variable assignments
    for i, input_var in enumerate(reversed(inputs)):
        cuda_code += f"        {input_var} = (x >> {i}) & 1;\n"
    
    cuda_code += "\n        // Evaluate circuit in topological order\n"
    # Generate equations in topological order
    for node in ordered_nodes:
        if node not in inputs:  # Skip input nodes
            input_order = G.nodes[node].get('input_order', [])
            truth_value = G.nodes[node].get('truth_value')
            
            if truth_value is not None and input_order:
                shifts = []
                for i, input_var in enumerate(input_order):
                    shifts.append(f"({input_var} << {len(input_order) - 1 - i})")
                equation = f"        {node} = {truth_value} >> (" + " + ".join(shifts) + ") & 1;\n"
                cuda_code += equation
    
    cuda_code += "\n        // Combine outputs into a single integer\n"
    cuda_code += "        output[idx] = 0;\n"
    
    # Combine outputs
    for i, output in enumerate(reversed(outputs)):
        cuda_code += f"        output[idx] |= ({output} << {i});\n"

    # i = 0
    # for node in reversed(ordered_nodes):
    #     if node in outputs:
    #         cuda_code += f"        output[idx] |= ({node} << {i});\n"
    #         i += 1

    
    cuda_code += "    }\n}\n\n"
    
    # Add main function with timing and validation
    cuda_code += """
int main() {
    // Calculate number of input combinations
    const int num_inputs = """ + str(num_inputs) + """;
    const int num_combinations = 1 << num_inputs;
    const int output_size = num_combinations * sizeof(int);
    
    // Create CUDA events for timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    // Allocate host memory
    int *h_output = (int*)malloc(output_size);
    
    // Allocate device memory
    int *d_output;
    CHECK_CUDA_ERROR(cudaMalloc(&d_output, output_size));
    
    // Calculate grid and block dimensions
    int threadsPerBlock = 256;
    int blocksPerGrid = (num_combinations + threadsPerBlock - 1) / threadsPerBlock;
    
    // Record start time
    cudaEventRecord(start);
    
    // Launch kernel
    evaluate_circuit<<<blocksPerGrid, threadsPerBlock>>>(d_output);
    CHECK_CUDA_ERROR(cudaGetLastError());
    
    // Record stop time
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    
    // Calculate elapsed time
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    
    // Copy result back to host
    CHECK_CUDA_ERROR(cudaMemcpy(h_output, d_output, output_size, cudaMemcpyDeviceToHost));
    
    // Print timing information
    printf("\\nGPU Execution time: %.3f milliseconds\\n", milliseconds);
    
    // Print truth table header
    printf("\\nCircuit Truth Table and Validation:\\n");
    printf("Inputs");
    for (int i = 0; i < """ + str(num_inputs/3) + """; i++) {
        printf("\\t");
    }
    printf("-> Outputs|Valid?\\n");
    
    // Print results and validate
    for (int i = 0; i < num_combinations; i++) {
        // Get input values
        int input1 = 0;
        int input2 = 0;
        
        // Extract first half of inputs for input1
        for (int j = num_inputs/2 - 1; j >= 0; j--) {
            input1 = (input1 << 1) | ((i >> (j + num_inputs/2)) & 1);
        }
        
        // Extract second half of inputs for input2
        for (int j = num_inputs/2 - 1; j >= 0; j--) {
            input2 = (input2 << 1) | ((i >> j) & 1);
        }
        
        // Calculate expected sum
        int expected_sum = input1 + input2;
        
        // Get actual output
        int actual_output = h_output[i];
        
        // Print input combinations
        for (int j = num_inputs - 1; j >= 0; j--) {
            printf("%d ", (i >> j) & 1);
        }
        printf("\\t-> ");
        
        // Print output values
        for (int j = """ + str(num_outputs - 1) + """; j >= 0; j--) {
            printf("%d", (actual_output >> j) & 1);
        }
        
        // Validate and print result
        if (actual_output != expected_sum) {
            printf(" | INVALID! Expected: %d, Got: %d", expected_sum, actual_output);
        } else {
            printf(" | Valid");
        }
        printf("\\n");
    }
    
    // Cleanup
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    free(h_output);
    cudaFree(d_output);
    
    return 0;
}
"""
    return cuda_code

def save_cuda_code(cuda_code, filename="circuit_evaluator.cu"):
    """Saves the CUDA code to a file."""
    with open(filename, "w") as f:
        f.write(cuda_code)
    print(f"CUDA code saved to {filename}")

# Example usage

cuda_code = generate_cuda_code(graph)

if cuda_code:
    save_cuda_code(cuda_code)
    print("\nTo compile and run:")
    print("1. nvcc circuit_evaluator.cu -o circuit_evaluator")
    print("2. ./circuit_evaluator")

CUDA code saved to circuit_evaluator.cu

To compile and run:
1. nvcc circuit_evaluator.cu -o circuit_evaluator
2. ./circuit_evaluator


In [107]:
# Run CUDA code
os.system("nvcc circuit_evaluator.cu -o circuit_evaluator")
os.system("./circuit_evaluator")


GPU Execution time: 0.190 milliseconds

Circuit Truth Table and Validation:
Inputs		-> Outputs|Valid?
0 0 0 0 0 0 	-> 0000 | Valid
0 0 0 0 0 1 	-> 0011 | INVALID! Expected: 1, Got: 3
0 0 0 0 1 0 	-> 0000 | INVALID! Expected: 2, Got: 0
0 0 0 0 1 1 	-> 0011 | Valid
0 0 0 1 0 0 	-> 0100 | Valid
0 0 0 1 0 1 	-> 0111 | INVALID! Expected: 5, Got: 7
0 0 0 1 1 0 	-> 0100 | INVALID! Expected: 6, Got: 4
0 0 0 1 1 1 	-> 0111 | Valid
0 0 1 0 0 0 	-> 0011 | INVALID! Expected: 1, Got: 3
0 0 1 0 0 1 	-> 0100 | INVALID! Expected: 2, Got: 4
0 0 1 0 1 0 	-> 0011 | Valid
0 0 1 0 1 1 	-> 0100 | Valid
0 0 1 1 0 0 	-> 0111 | INVALID! Expected: 5, Got: 7
0 0 1 1 0 1 	-> 0000 | INVALID! Expected: 6, Got: 0
0 0 1 1 1 0 	-> 0111 | Valid
0 0 1 1 1 1 	-> 0000 | INVALID! Expected: 8, Got: 0
0 1 0 0 0 0 	-> 0000 | INVALID! Expected: 2, Got: 0
0 1 0 0 0 1 	-> 0011 | Valid
0 1 0 0 1 0 	-> 0010 | INVALID! Expected: 4, Got: 2
0 1 0 0 1 1 	-> 0101 | Valid
0 1 0 1 0 0 	-> 0100 | INVALID! Expected: 6, Got: 4
0 1 0 1 0 1 

0